# Notebook 5 v2 — Feature comparison and cross-scenario generalisation

**Question:** do feature sets A/F/G/H generalise when event direction or drift type changes, and how much of any failure is attributable to distribution shift?

## Design

- **Input:** `data/processed/feature_dataset_AF_v2.parquet`, produced by NB4 v2. The v1 files are not modified.
- **Rows:** pure windows are primary; transition windows are evaluated separately and never used for fitting/tuning.
- **Feature sets:** A (16 original), F (15 revised), G (F without `peer_level_rel`, 14), H (A + `agree_sign` + `dev_mean_adj`, 18).
- **Training regimes:**
  - **W (warming/ramp only):** train and tune only on the four v1 variants (`NORMAL`, `DRIFT_RAMP`, `EVENT_WARM`, `ED_WARM_RAMP`). Cooling and step variants are never used for fitting or hyperparameter selection.
  - **ALL (all variants):** train/tune on all nine variants. This is a learnability/reference condition, **not** an out-of-distribution test.
- **Test conditions:** baseline, `cool_ramp`, `warm_step`, and `cool_step`. Each condition contains one variant for each of the four labels. The same held-out test episodes are used in every condition.
- **Model:** Random Forest, balanced class weights, 300 trees, 5 seeds. Hyperparameters are selected on validation macro-F1 separately for each feature set and training regime. Test episodes are not used for selection.
- **Imputation:** median learned from the fitting rows only, separately for each fit.
- **Robustness:** leave-one-episode-out (LOEO) on all 18 episodes, with a fixed predeclared forest configuration (150 trees, `min_samples_leaf=5`, `max_features='sqrt'`) to avoid reusing validation-selected hyperparameters on episodes that served as validation. LOEO is a robustness analysis, not a substitute for the held-out split.

**Interpretation guardrails:** windows overlap heavily; only 4 episodes are in the held-out test split. Cross-condition test sets reuse NORMAL or DRIFT control variants where the changed factor does not apply. The ALL regime has seen all variant types during training, so only W measures cross-direction/cross-drift-type generalisation. Step drift has a larger average effect than ramp drift by construction; do not interpret that comparison as drift-type alone without noting magnitude/profile differences.

This notebook reports evidence from the synthetic scenario generator. It does not establish real-world performance, calibration, or safe correction.

In [2]:
from pathlib import Path
import json, warnings, hashlib
import numpy as np
import pandas as pd

try:
    import sklearn
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
except ImportError as e:
    raise ImportError("scikit-learn is required. Install it in this Jupyter kernel and restart the kernel.") from e

_here = Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "data" / "processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find data/processed above " + str(_here) + ". Start Jupyter inside the project folder.")
PROC = ROOT / "data" / "processed"
OUT = ROOT / "outputs" / "notebook05_v2"
OUT.mkdir(parents=True, exist_ok=True)

INPUT = PROC / "feature_dataset_AF_v2.parquet"
CONFIG = PROC / "feature_config_AF_v2.json"
if not INPUT.exists() or not CONFIG.exists():
    raise FileNotFoundError(f"Run NB4 v2 first. Required files: {INPUT.name}, {CONFIG.name}")

def check(cond, msg):
    if not bool(cond):
        raise ValueError("CHECK FAILED: " + msg)
    print("PASS ", msg)

def warn(cond, msg):
    print("OK   " if cond else "WARN ", msg)

print("ROOT:", ROOT, "| sklearn:", sklearn.__version__, "| pandas:", pd.__version__)
print("Output directory:", OUT)

ROOT: C:\AIoT_Research | sklearn: 1.9.1 | pandas: 2.2.2
Output directory: C:\AIoT_Research\outputs\notebook05_v2


## 1. Load data and define feature sets

The condition definitions use the `variant` column, not class labels alone. Every condition must map to exactly one variant for each of the four classes.

In [4]:
LABELS = ["NORMAL", "DRIFT", "EVENT", "EVENT_DRIFT"]
SEEDS = [0, 1, 2, 3, 4]
N_TREES = 300
N_TREES_LOEO = 150
GRID = [{"min_samples_leaf": m, "max_features": mf} for m in (1, 5, 20) for mf in ("sqrt", 0.5)]
N_BOOT = 2000

with open(CONFIG, encoding="utf-8") as f:
    FC = json.load(f)
T = pd.read_parquet(INPUT)
A_FEATURES = list(FC["A_features"])
F_FEATURES = list(FC["F_features"])
G_FEATURES = [f for f in F_FEATURES if f != "peer_level_rel"]
H_FEATURES = A_FEATURES + ["agree_sign", "dev_mean_adj"]
ALL_FEATURES = list(FC["all_features"])
SETS = {"A": A_FEATURES, "F": F_FEATURES, "G": G_FEATURES, "H": H_FEATURES}

BASELINE = ["NORMAL", "DRIFT_RAMP", "EVENT_WARM", "ED_WARM_RAMP"]
CONDITIONS = {
    "baseline": BASELINE,
    "cool_ramp": ["NORMAL", "DRIFT_RAMP", "EVENT_COOL", "ED_COOL_RAMP"],
    "warm_step": ["NORMAL", "DRIFT_STEP", "EVENT_WARM", "ED_WARM_STEP"],
    "cool_step": ["NORMAL", "DRIFT_STEP", "EVENT_COOL", "ED_COOL_STEP"],
}
VARIANT_TO_LABEL = {
    "NORMAL": "NORMAL", "DRIFT_RAMP": "DRIFT", "DRIFT_STEP": "DRIFT",
    "EVENT_WARM": "EVENT", "EVENT_COOL": "EVENT",
    "ED_WARM_RAMP": "EVENT_DRIFT", "ED_WARM_STEP": "EVENT_DRIFT",
    "ED_COOL_RAMP": "EVENT_DRIFT", "ED_COOL_STEP": "EVENT_DRIFT",
}
W_VARIANTS = set(BASELINE)
ALL_VARIANTS = set(FC.get("variants", T["variant"].unique().tolist()))
REGIMES = {"W": W_VARIANTS, "ALL": ALL_VARIANTS}

check(len(A_FEATURES) == 16 and len(F_FEATURES) == 15 and len(G_FEATURES) == 14 and len(H_FEATURES) == 18,
      "feature set sizes are A=16, F=15, G=14, H=18")
check(set(H_FEATURES) == set(A_FEATURES) | {"agree_sign", "dev_mean_adj"} and "peer_level_rel" not in H_FEATURES,
      "H = A + agree_sign + dev_mean_adj, without peer_level_rel")
check(set(G_FEATURES) == set(F_FEATURES) - {"peer_level_rel"}, "G = F without peer_level_rel")
check(set(ALL_FEATURES) <= set(T.columns), "all expected feature columns exist")
check(set(T["label"].dropna().unique()) == set(LABELS), "four expected labels are present")
check(set(T["variant"].unique()) == ALL_VARIANTS, "all nine variants are present")
check(T["window_id"].is_unique, "window_id is unique")

LEAK = {"label", "scenario", "episode_id", "split", "variant", "event_direction", "drift_type", "drift_event_relation",
        "interval_phase", "label_purity", "long_label_purity", "window_pure", "long_window_pure", "window_group",
        "minutes_since_onset", "gt_event_component", "gt_drift_component", "offset_used", "baseline_used", "end_index"}
check(all(not (LEAK & set(feats)) for feats in SETS.values()), "no label/ground-truth/split metadata is used as a feature")

# The notebook uses the explicit window_group label generated by NB4 v2.
if "window_group" not in T.columns:
    raise KeyError("NB4 v2 table has no window_group column")
T["window_pure"] = T["window_group"].eq("pure")
T["label"] = T["label"].astype(str)
print("Rows:", len(T), "| episodes:", T["episode_id"].nunique(), "| split counts:", T["split"].value_counts().to_dict())
print("Pure/transition:", T["window_pure"].value_counts().to_dict())
print("Episodes per split:", T.groupby("split")["episode_id"].nunique().to_dict())
print("Feature counts:", {k: len(v) for k, v in SETS.items()})

PASS  feature set sizes are A=16, F=15, G=14, H=18
PASS  H = A + agree_sign + dev_mean_adj, without peer_level_rel
PASS  G = F without peer_level_rel
PASS  all expected feature columns exist
PASS  four expected labels are present
PASS  all nine variants are present
PASS  window_id is unique
PASS  no label/ground-truth/split metadata is used as a feature
Rows: 11826 | episodes: 18 | split counts: {'train': 6570, 'validation': 2628, 'test': 2628}
Pure/transition: {True: 11106, False: 720}
Episodes per split: {'test': 4, 'train': 10, 'validation': 4}
Feature counts: {'A': 16, 'F': 15, 'G': 14, 'H': 18}


## 2. Data and scenario checks

These checks catch a stale v1 feature table, missing variants, split overlap, or a condition definition that does not cover all four classes. They do not claim that the synthetic generator is realistic.

In [6]:
check(T.groupby("episode_id")["variant"].nunique().eq(9).all(), "each episode has all 9 variants")
eps_by_split = T.groupby("split")["episode_id"].apply(lambda x: set(x))
split_names = list(eps_by_split.index)
check(all(eps_by_split[a].isdisjoint(eps_by_split[b]) for i, a in enumerate(split_names) for b in split_names[i+1:]),
      "episodes do not cross train/validation/test splits")
for cond, variants in CONDITIONS.items():
    check(len(variants) == 4 and set(VARIANT_TO_LABEL[v] for v in variants) == set(LABELS), f"{cond}: exactly one variant per class")
    check(set(variants) <= ALL_VARIANTS, f"{cond}: all variants exist in the feature table")

PURE = T[T["window_pure"]].copy()
TRANS = T[~T["window_pure"]].copy()
for split in ["train", "validation", "test"]:
    part = PURE[PURE["split"] == split]
    print(f"\nPure {split} rows by label:")
    print(part["label"].value_counts().reindex(LABELS, fill_value=0).to_string())
    check(set(part["label"].unique()) == set(LABELS), f"all four classes appear in pure {split} windows")

print("\nPure rows by split and variant:")
print(PURE.groupby(["split", "variant"]).size().unstack(fill_value=0).to_string())
print("\nMissing feature values (% of all windows):")
print((100 * T[ALL_FEATURES].isna().mean()).round(2).to_string())

PASS  each episode has all 9 variants
PASS  episodes do not cross train/validation/test splits
PASS  baseline: exactly one variant per class
PASS  baseline: all variants exist in the feature table
PASS  cool_ramp: exactly one variant per class
PASS  cool_ramp: all variants exist in the feature table
PASS  warm_step: exactly one variant per class
PASS  warm_step: all variants exist in the feature table
PASS  cool_step: exactly one variant per class
PASS  cool_step: all variants exist in the feature table

Pure train rows by label:
label
NORMAL          810
DRIFT          1340
EVENT          1340
EVENT_DRIFT    2680
PASS  all four classes appear in pure train windows

Pure validation rows by label:
label
NORMAL          324
DRIFT           536
EVENT           536
EVENT_DRIFT    1072
PASS  all four classes appear in pure validation windows

Pure test rows by label:
label
NORMAL          324
DRIFT           536
EVENT           536
EVENT_DRIFT    1072
PASS  all four classes appear in pure t

## 3. Fit/predict helpers

Imputation medians are fitted on the fitting rows only. All test-condition scores are produced after hyperparameters are selected on validation.

In [8]:
def get_split_rows(split, variants=None, pure=True, source=PURE):
    d = source[(source["split"] == split) & (source["window_pure"] == pure)]
    if variants is not None:
        d = d[d["variant"].isin(variants)]
    return d.copy()

def to_labels(prob_df):
    return np.array(LABELS)[prob_df[LABELS].to_numpy().argmax(axis=1)]

def fit_predict(feats, params, seed, train_df, eval_dfs, n_trees=N_TREES):
    med = train_df[feats].median()  # fitted only on training rows
    if med.isna().any():
        missing = med[med.isna()].index.tolist()
        raise ValueError(f"Feature(s) have no observed training values: {missing}")
    Xtr = train_df[feats].fillna(med).to_numpy(dtype=float)
    rf = RandomForestClassifier(n_estimators=n_trees, class_weight="balanced", random_state=seed,
                                n_jobs=-1, **params)
    rf.fit(Xtr, train_df["label"].to_numpy())
    if set(rf.classes_) != set(LABELS):
        raise ValueError(f"Training data does not contain all labels: {rf.classes_}")
    out = {}
    for name, df in eval_dfs.items():
        X = df[feats].fillna(med).to_numpy(dtype=float)
        P = rf.predict_proba(X)
        tmp = pd.DataFrame(P, columns=rf.classes_, index=df.index)
        out[name] = tmp[LABELS]
    return out

def metric_dict(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    out = {"accuracy": float(accuracy_score(y_true, y_pred)), "macro_f1": float(f.mean())}
    for i, lab in enumerate(LABELS):
        out[f"precision_{lab}"] = float(p[i])
        out[f"recall_{lab}"] = float(r[i])
        out[f"f1_{lab}"] = float(f[i])
    return out

def select_rows_for_condition(df, condition):
    return df[df["variant"].isin(CONDITIONS[condition])].copy()

def ensemble_prob(prob_list):
    return sum(prob_list) / len(prob_list)

print("Model helpers defined.")

Model helpers defined.


## 4. Validation-only hyperparameter selection

Each feature set and training regime gets its own selection using validation macro-F1 averaged over the five fixed seeds. Regime W sees only baseline/warming-ramp variants in both training and validation. Regime ALL sees all nine variants in both.

In [10]:
BEST = {}
SEL_ROWS = []
VAL_PROBS = {}

for regime, allowed in REGIMES.items():
    tr = get_split_rows("train", allowed)
    va = get_split_rows("validation", allowed)
    check(len(tr) > 0 and len(va) > 0, f"regime {regime}: train and validation rows available")
    check(set(tr["episode_id"]).isdisjoint(set(va["episode_id"])), f"regime {regime}: train/validation episodes are disjoint")
    check(set(tr["variant"]) <= allowed and set(va["variant"]) <= allowed, f"regime {regime}: no excluded variants used for tuning")
    for set_name, feats in SETS.items():
        scores = []
        for params in GRID:
            seed_scores = []
            for seed in SEEDS:
                P = fit_predict(feats, params, seed, tr, {"validation": va})["validation"]
                VAL_PROBS[(regime, set_name, tuple(sorted(params.items())), seed)] = P
                seed_scores.append(f1_score(va["label"], to_labels(P), average="macro", labels=LABELS, zero_division=0))
            mean_score = float(np.mean(seed_scores))
            scores.append((mean_score, params.copy()))
            SEL_ROWS.append({"regime": regime, "set": set_name, **params, "validation_macro_f1_mean5": mean_score,
                             "validation_macro_f1_sd5": float(np.std(seed_scores, ddof=1))})
        # Deterministic tie-break: first item in the predeclared grid wins.
        best_score, best_params = max(scores, key=lambda z: z[0])
        BEST[(regime, set_name)] = best_params
        print(f"{regime}/{set_name}: {best_params} | validation macro-F1={best_score:.3f}")

SEL = pd.DataFrame(SEL_ROWS)
print("\nValidation selection table:")
print(SEL.sort_values(["regime", "set", "validation_macro_f1_mean5"], ascending=[True, True, False]).to_string(index=False))

PASS  regime W: train and validation rows available
PASS  regime W: train/validation episodes are disjoint
PASS  regime W: no excluded variants used for tuning
W/A: {'min_samples_leaf': 20, 'max_features': 'sqrt'} | validation macro-F1=0.533
W/F: {'min_samples_leaf': 1, 'max_features': 'sqrt'} | validation macro-F1=0.590
W/G: {'min_samples_leaf': 20, 'max_features': 0.5} | validation macro-F1=0.532
W/H: {'min_samples_leaf': 1, 'max_features': 'sqrt'} | validation macro-F1=0.493
PASS  regime ALL: train and validation rows available
PASS  regime ALL: train/validation episodes are disjoint
PASS  regime ALL: no excluded variants used for tuning
ALL/A: {'min_samples_leaf': 20, 'max_features': 'sqrt'} | validation macro-F1=0.547
ALL/F: {'min_samples_leaf': 5, 'max_features': 'sqrt'} | validation macro-F1=0.565
ALL/G: {'min_samples_leaf': 5, 'max_features': 'sqrt'} | validation macro-F1=0.561
ALL/H: {'min_samples_leaf': 20, 'max_features': 'sqrt'} | validation macro-F1=0.543

Validation selec

## 5. Held-out test: baseline, cooling, step, and cooling+step

The test split is scored only after selection is complete. The **W** regime is the actual cross-condition generalisation test. **ALL** is a comparison showing performance when the model has examples of all scenario types during training.

In [16]:
TEST_PROBS = {}  # (regime, set, condition, seed) -> probability dataframe
TEST_ROWS = []
CM_ROWS = []
PRED_FRAMES = []

for regime, allowed in REGIMES.items():
    tr = get_split_rows("train", allowed)
    # For W, this remains the four original warming/ramp variants; ALL sees all nine.
    for set_name, feats in SETS.items():
        params = BEST[(regime, set_name)]
        for seed in SEEDS:
            # Fit once per regime/set/seed; evaluate the four condition subsets using the same fitted model.
            test_all = get_split_rows("test")
            P_by_condition = {}
            med = tr[feats].median()
            if med.isna().any():
                raise ValueError(f"{regime}/{set_name}: no train observations for {med[med.isna()].index.tolist()}")
            rf = RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced", random_state=seed,
                                        n_jobs=-1, **params)
            rf.fit(tr[feats].fillna(med).to_numpy(float), tr["label"].to_numpy())
            check_classes = set(rf.classes_) == set(LABELS)
            if not check_classes:
                raise ValueError(f"{regime}/{set_name}: missing class in training")
            for condition, variants in CONDITIONS.items():
                te = test_all[test_all["variant"].isin(variants)].copy()
                # Exactly one scenario variant per class, and all four classes should occur.
                if set(te["variant"].unique()) != set(variants) or set(te["label"].unique()) != set(LABELS):
                    raise ValueError(f"Test condition {condition} is incomplete for {regime}/{set_name}")
                probs = rf.predict_proba(te[feats].fillna(med).to_numpy(float))
                P = pd.DataFrame(probs, columns=rf.classes_, index=te.index)[LABELS]
                TEST_PROBS[(regime, set_name, condition, seed)] = P
                pred = to_labels(P)
                mdict = metric_dict(te["label"].to_numpy(), pred)
                TEST_ROWS.append({"regime": regime, "set": set_name, "condition": condition, "seed": seed,
                                  "n_windows": len(te), "n_episodes": te["episode_id"].nunique(), **mdict})
                cm = confusion_matrix(te["label"], pred, labels=LABELS)
                for i, actual in enumerate(LABELS):
                    for j, predicted in enumerate(LABELS):
                        CM_ROWS.append({"regime": regime, "set": set_name, "condition": condition, "seed": seed,
                                        "actual": actual, "predicted": predicted, "count": int(cm[i, j])})
                pred_frame = te[["window_id", "episode_id", "split", "variant", "label", "window_group", "window_end",
                                 "event_direction", "drift_type", "drift_event_relation", "interval_phase", "minutes_since_onset",
                                 "gt_event_component", "gt_drift_component"]].copy()
                pred_frame["regime"] = regime; pred_frame["set"] = set_name; pred_frame["condition"] = condition; pred_frame["seed"] = seed
                pred_frame["predicted"] = pred
                for lab in LABELS:
                    pred_frame[f"p_{lab}"] = P[lab].to_numpy()
                PRED_FRAMES.append(pred_frame)

TEST = pd.DataFrame(TEST_ROWS)
CM = pd.DataFrame(CM_ROWS)
PRED = pd.concat(PRED_FRAMES, ignore_index=True)
print("Test prediction rows:", len(PRED), "| metric records:", len(TEST))

Test prediction rows: 177280 | metric records: 160


## 6. Primary results and confusion matrices

Metrics are first shown as mean ± standard deviation across five seeds. Seeds measure algorithmic variability, **not** independent data replications. Macro-F1 is the main metric; EVENT_DRIFT recall is highlighted because it is the central attribution challenge.

In [18]:
METRIC_COLS = ["accuracy", "macro_f1", "precision_EVENT", "recall_EVENT", "f1_EVENT",
               "precision_DRIFT", "recall_DRIFT", "f1_DRIFT",
               "precision_EVENT_DRIFT", "recall_EVENT_DRIFT", "f1_EVENT_DRIFT"]
def mean_sd(s):
    return f"{s.mean():.3f} ± {s.std(ddof=1):.3f}" if len(s) > 1 else f"{s.mean():.3f}"
SUMMARY = TEST.groupby(["regime", "condition", "set"])[METRIC_COLS].agg(["mean", "std"])
print("Mean ± SD across seeds; regime W is the cross-condition generalisation test:")
for regime in ["W", "ALL"]:
    print(f"\n===== REGIME {regime} =====")
    for condition in CONDITIONS:
        sub = TEST[(TEST["regime"] == regime) & (TEST["condition"] == condition)]
        rows = []
        for set_name in SETS:
            d = sub[sub["set"] == set_name]
            rows.append({"set": set_name, **{c: mean_sd(d[c]) for c in METRIC_COLS}})
        print(f"\nCondition: {condition}")
        print(pd.DataFrame(rows).set_index("set").to_string())

print("\nConfusion matrices averaged across seeds (actual rows, predicted columns):")
for regime in ["W", "ALL"]:
    for condition in CONDITIONS:
        print(f"\n{regime} / {condition}")
        for set_name in SETS:
            d = CM[(CM["regime"] == regime) & (CM["condition"] == condition) & (CM["set"] == set_name)]
            mat = d.groupby(["actual", "predicted"])["count"].mean().unstack(fill_value=0).reindex(index=LABELS, columns=LABELS, fill_value=0)
            print(f"{set_name}:\n{mat.round(1).to_string()}")

Mean ± SD across seeds; regime W is the cross-condition generalisation test:

===== REGIME W =====

Condition: baseline
          accuracy       macro_f1 precision_EVENT   recall_EVENT       f1_EVENT precision_DRIFT   recall_DRIFT       f1_DRIFT precision_EVENT_DRIFT recall_EVENT_DRIFT f1_EVENT_DRIFT
set                                                                                                                                                                                  
A    0.579 ± 0.009  0.580 ± 0.009   0.449 ± 0.013  0.707 ± 0.014  0.549 ± 0.014   0.730 ± 0.012  0.596 ± 0.006  0.656 ± 0.006         0.567 ± 0.009      0.440 ± 0.025  0.495 ± 0.019
F    0.628 ± 0.004  0.626 ± 0.004   0.508 ± 0.015  0.728 ± 0.009  0.599 ± 0.008   0.700 ± 0.021  0.728 ± 0.010  0.713 ± 0.014         0.581 ± 0.006      0.442 ± 0.011  0.502 ± 0.009
G    0.486 ± 0.005  0.485 ± 0.004   0.381 ± 0.004  0.628 ± 0.006  0.475 ± 0.004   0.624 ± 0.011  0.410 ± 0.022  0.495 ± 0.016         0.444 ± 0.009     

## 7. Condition-level changes and per-variant correct rates

The W regime's change from baseline to `cool_ramp` is the cleanest available direction diagnostic. The change to step conditions is harder to interpret as a pure drift-type effect because step offsets have a larger average magnitude than ramps by design.

In [20]:
# Average probabilities over seeds, then make one ensemble prediction per test window/condition.
ENSEMBLE_ROWS = []
for regime in REGIMES:
    for set_name in SETS:
        for condition, variants in CONDITIONS.items():
            te = get_split_rows("test", variants)
            plist = [TEST_PROBS[(regime, set_name, condition, seed)].reindex(te.index) for seed in SEEDS]
            Pm = ensemble_prob(plist)
            pred = to_labels(Pm)
            d = te.copy(); d["predicted"] = pred; d["correct"] = d["predicted"].eq(d["label"])
            d["regime"] = regime; d["set"] = set_name; d["condition"] = condition
            ENSEMBLE_ROWS.append(d)
ENSEMBLE = pd.concat(ENSEMBLE_ROWS, ignore_index=True)

variant_acc = ENSEMBLE.groupby(["regime", "set", "condition", "variant"]).agg(
    n=("label", "size"), accuracy=("correct", "mean")).reset_index()
print("Per-variant correct rate (ensemble over seeds):")
print(variant_acc.pivot_table(index=["regime", "set", "condition"], columns="variant", values="accuracy").round(3).to_string())

# Paired condition deltas on the same held-out episodes, computed episode-by-episode.
EPISODE_METRICS = []
for (regime, set_name, condition, episode), d in ENSEMBLE.groupby(["regime", "set", "condition", "episode_id"]):
    mm = metric_dict(d["label"].to_numpy(), d["predicted"].to_numpy())
    EPISODE_METRICS.append({"regime": regime, "set": set_name, "condition": condition, "episode_id": episode,
                            "n": len(d), "macro_f1": mm["macro_f1"], "accuracy": mm["accuracy"],
                            "recall_EVENT_DRIFT": mm["recall_EVENT_DRIFT"]})
EPM = pd.DataFrame(EPISODE_METRICS)
DELTA_ROWS = []
rng = np.random.default_rng(20261018)
for regime in REGIMES:
    for set_name in SETS:
        for condition in ["cool_ramp", "warm_step", "cool_step"]:
            a = EPM[(EPM.regime == regime) & (EPM.set == set_name) & (EPM.condition == "baseline")].set_index("episode_id")
            b = EPM[(EPM.regime == regime) & (EPM.set == set_name) & (EPM.condition == condition)].set_index("episode_id")
            joined = a[["macro_f1", "recall_EVENT_DRIFT"]].join(b[["macro_f1", "recall_EVENT_DRIFT"]], lsuffix="_base", rsuffix="_cond", how="inner")
            for metric in ["macro_f1", "recall_EVENT_DRIFT"]:
                diff = (joined[f"{metric}_cond"] - joined[f"{metric}_base"]).to_numpy(float)
                if len(diff) == 0:
                    continue
                boot = np.array([rng.choice(diff, size=len(diff), replace=True).mean() for _ in range(N_BOOT)])
                DELTA_ROWS.append({"regime": regime, "set": set_name, "condition_vs_baseline": condition, "metric": metric,
                                   "mean_paired_delta": float(diff.mean()), "ci_low": float(np.percentile(boot, 2.5)),
                                   "ci_high": float(np.percentile(boot, 97.5)),
                                   "episodes_positive": int((diff > 0).sum()), "episodes_negative": int((diff < 0).sum()),
                                   "n_episodes": int(len(diff))})
DELTA = pd.DataFrame(DELTA_ROWS)
print("\nPaired per-episode change vs baseline; bootstrap resamples episodes (4 test episodes, so intervals are unstable):")
print(DELTA.round(3).to_string(index=False))

Per-variant correct rate (ensemble over seeds):
variant               DRIFT_RAMP  DRIFT_STEP  ED_COOL_RAMP  ED_COOL_STEP  ED_WARM_RAMP  ED_WARM_STEP  EVENT_COOL  EVENT_WARM  NORMAL
regime set condition                                                                                                                
ALL    A   baseline        0.346         NaN           NaN           NaN         0.272           NaN         NaN       0.783   0.312
           cool_ramp       0.346         NaN         0.555           NaN           NaN           NaN       0.838         NaN   0.312
           cool_step         NaN       0.665           NaN         0.853           NaN           NaN       0.838         NaN   0.312
           warm_step         NaN       0.665           NaN           NaN           NaN         0.684         NaN       0.783   0.312
       F   baseline        0.430         NaN           NaN           NaN         0.463           NaN         NaN       0.787   0.466
           cool_ramp 

## 8. Leave-one-episode-out robustness check across all 18 episodes

For each held-out episode, fit on the other 17 episodes and score the four conditions within the held-out episode. The W regime trains only on the baseline variants; ALL trains on all nine variants. A fixed, predeclared model configuration is used here, rather than validation-selected hyperparameters, so the episode being held out cannot have influenced model selection through the standard validation split.

In [30]:
LOEO_PARAMS = {"min_samples_leaf": 5, "max_features": "sqrt"}
LOEO_ROWS = []
PURE_ALL = PURE.copy()
episodes = sorted(PURE_ALL["episode_id"].unique())
check(len(episodes) == 18, f"LOEO uses all 18 episodes (found {len(episodes)})")

for heldout in episodes:
    train_eps = PURE_ALL[PURE_ALL["episode_id"] != heldout]
    held_eps = PURE_ALL[PURE_ALL["episode_id"] == heldout]
    for regime, allowed in REGIMES.items():
        tr = train_eps[train_eps["variant"].isin(allowed)]
        for set_name, feats in SETS.items():
            med = tr[feats].median()
            if med.isna().any():
                raise ValueError(f"LOEO {regime}/{set_name}/{heldout}: no train values for {med[med.isna()].index.tolist()}")
            rf = RandomForestClassifier(n_estimators=N_TREES_LOEO, class_weight="balanced", random_state=0,
                                        n_jobs=-1, **LOEO_PARAMS)
            rf.fit(tr[feats].fillna(med).to_numpy(float), tr["label"].to_numpy())
            if set(rf.classes_) != set(LABELS):
                raise ValueError(f"LOEO training missing classes for {regime}/{set_name}/{heldout}")
            for condition, variants in CONDITIONS.items():
                ho = held_eps[held_eps["variant"].isin(variants)]
                P = pd.DataFrame(rf.predict_proba(ho[feats].fillna(med).to_numpy(float)), columns=rf.classes_, index=ho.index)[LABELS]
                pred = to_labels(P)
                mm = metric_dict(ho["label"].to_numpy(), pred)
                ed = (ho["label"].to_numpy() == "EVENT_DRIFT")
                ed_big = ed & (ho["gt_drift_component"].abs().to_numpy() >= 1.5)
                LOEO_ROWS.append({"heldout_episode": heldout, "original_split": str(ho["split"].iloc[0]),
                                  "regime": regime, "set": set_name, "condition": condition, "n": len(ho),
                                  "macro_f1": mm["macro_f1"], "accuracy": mm["accuracy"],
                                  "recall_EVENT_DRIFT": mm["recall_EVENT_DRIFT"],
                                  "recall_EVENT_DRIFT_absdrift_ge1p5": float((pred[ed_big] == "EVENT_DRIFT").mean()) if ed_big.any() else np.nan})

LOEO = pd.DataFrame(LOEO_ROWS)
print("LOEO records:", len(LOEO), "| episodes:", LOEO["heldout_episode"].nunique())
print("\nMean ± SD across held-out episodes:")
print(LOEO.groupby(["regime", "condition", "set"])[["macro_f1", "accuracy", "recall_EVENT_DRIFT", "recall_EVENT_DRIFT_absdrift_ge1p5"]].agg(["mean", "std"]).round(3).to_string())

# Paired bootstrap of episode-level differences, using identical held-out episodes.
LOEO_DELTA_ROWS = []
rng = np.random.default_rng(42)

for (regime, condition), d in LOEO.groupby(["regime", "condition"]):
    wide = d.pivot(
        index="heldout_episode",
        columns="set",
        values="macro_f1"
    )

    for comparator in ["F", "G", "H"]:
        diff = (wide[comparator] - wide["A"]).dropna().to_numpy(float)
        if len(diff) == 0:
            continue

        # Vectorised bootstrap: much faster than a Python loop.
        samples = rng.choice(
            diff, size=(N_BOOT, len(diff)), replace=True
        )
        boot_means = samples.mean(axis=1)

        LOEO_DELTA_ROWS.append({
            "regime": regime,
            "condition": condition,
            "comparison": f"{comparator}-A",
            "mean_macro_f1_delta": float(diff.mean()),
            "ci_low": float(np.percentile(boot_means, 2.5)),
            "ci_high": float(np.percentile(boot_means, 97.5)),
            "episodes_better": int((diff > 0).sum()),
            "episodes_worse": int((diff < 0).sum()),
            "n_episodes": len(diff),
        })

LOEO_DELTA = pd.DataFrame(LOEO_DELTA_ROWS)
print(LOEO_DELTA.round(3).to_string(index=False))

PASS  LOEO uses all 18 episodes (found 18)
LOEO records: 576 | episodes: 18

Mean ± SD across held-out episodes:
                     macro_f1        accuracy        recall_EVENT_DRIFT        recall_EVENT_DRIFT_absdrift_ge1p5       
                         mean    std     mean    std               mean    std                              mean    std
regime condition set                                                                                                   
ALL    baseline  A      0.500  0.195    0.525  0.190              0.499  0.215                             0.713  0.284
                 F      0.564  0.142    0.584  0.136              0.582  0.259                             0.811  0.243
                 G      0.498  0.143    0.523  0.132              0.560  0.255                             0.816  0.230
                 H      0.532  0.227    0.556  0.216              0.524  0.215                             0.760  0.268
       cool_ramp A      0.528  0.191    0.563  

## 9. Transition-window evaluation (secondary)

Transition windows were excluded from fitting and tuning. They are scored separately to show how results change when the short window contains mixed labels. Do not merge these with the pure-window primary results.

In [32]:
TRANS_ROWS = []
for regime, allowed in REGIMES.items():
    tr = get_split_rows("train", allowed, pure=True)
    for set_name, feats in SETS.items():
        params = BEST[(regime, set_name)]
        for condition, variants in CONDITIONS.items():
            te = get_split_rows("test", variants, pure=False, source=T)
            if te.empty:
                TRANS_ROWS.append({"regime": regime, "set": set_name, "condition": condition, "n": 0, "macro_f1": np.nan, "accuracy": np.nan})
                continue
            vals = []
            for seed in SEEDS:
                P = fit_predict(feats, params, seed, tr, {"transition_test": te})["transition_test"]
                pred = to_labels(P)
                mm = metric_dict(te["label"].to_numpy(), pred)
                vals.append(mm)
            TRANS_ROWS.append({"regime": regime, "set": set_name, "condition": condition, "n": len(te),
                               "macro_f1_mean5": float(np.mean([x["macro_f1"] for x in vals])),
                               "macro_f1_sd5": float(np.std([x["macro_f1"] for x in vals], ddof=1)),
                               "accuracy_mean5": float(np.mean([x["accuracy"] for x in vals])),
                               "accuracy_sd5": float(np.std([x["accuracy"] for x in vals], ddof=1))})
TRANS_RESULTS = pd.DataFrame(TRANS_ROWS)
print(TRANS_RESULTS.round(3).to_string(index=False))

regime set condition  n  macro_f1_mean5  macro_f1_sd5  accuracy_mean5  accuracy_sd5
     W   A  baseline 60           0.113         0.006           0.197         0.014
     W   A cool_ramp 60           0.133         0.000           0.267         0.000
     W   A warm_step 60           0.307         0.020           0.407         0.025
     W   A cool_step 60           0.376         0.012           0.523         0.015
     W   F  baseline 60           0.179         0.035           0.257         0.032
     W   F cool_ramp 60           0.193         0.029           0.333         0.024
     W   F warm_step 60           0.371         0.021           0.473         0.019
     W   F cool_step 60           0.414         0.021           0.557         0.028
     W   G  baseline 60           0.130         0.002           0.287         0.007
     W   G cool_ramp 60           0.131         0.001           0.300         0.000
     W   G warm_step 60           0.295         0.004           0.480       

## 10. Reproducibility, saved outputs, and conclusion guardrails

The outputs are written to a new directory; v1 NB5 outputs are not overwritten. The configuration records the input file hash and all protocol settings.

In [34]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

# Refit one selected model/seed and check its test probabilities are exactly reproducible.
regime, set_name, condition, seed = "W", "A", "baseline", 0
tr = get_split_rows("train", REGIMES[regime])
te = get_split_rows("test", CONDITIONS[condition])
P_again = fit_predict(SETS[set_name], BEST[(regime, set_name)], seed, tr, {"test": te})["test"]
P_saved = TEST_PROBS[(regime, set_name, condition, seed)].reindex(te.index)
check(np.allclose(P_again.to_numpy(), P_saved.to_numpy()), "refit reproduces W/A/baseline test probabilities exactly")

SEL.to_csv(OUT / "validation_model_selection.csv", index=False)
TEST.to_csv(OUT / "test_metrics_by_seed.csv", index=False)
CM.to_csv(OUT / "test_confusion_counts_by_seed.csv", index=False)
variant_acc.to_csv(OUT / "test_per_variant_correct_rate.csv", index=False)
EPM.to_csv(OUT / "test_per_episode_metrics.csv", index=False)
DELTA.to_csv(OUT / "test_paired_condition_deltas.csv", index=False)
LOEO.to_csv(OUT / "loeo_per_episode.csv", index=False)
LOEO_DELTA.to_csv(OUT / "loeo_paired_feature_deltas.csv", index=False)
TRANS_RESULTS.to_csv(OUT / "transition_window_metrics.csv", index=False)
PRED.to_parquet(OUT / "test_predictions_by_seed.parquet", index=False)
ENSEMBLE.to_parquet(OUT / "test_predictions_ensemble.parquet", index=False)

config_out = {
    "input_file": INPUT.name, "input_sha256": sha256(INPUT), "feature_config_file": CONFIG.name,
    "feature_sets": {k: v for k, v in SETS.items()}, "labels": LABELS,
    "conditions": CONDITIONS, "training_regimes": {k: sorted(v) for k, v in REGIMES.items()},
    "seeds": SEEDS, "n_trees_main": N_TREES, "grid": GRID,
    "chosen_hyperparameters": {f"{r}/{s}": p for (r, s), p in BEST.items()},
    "imputation": "median fit on the fitting rows only, separately per model fit",
    "primary_rows": "pure windows; transitions evaluated separately",
    "loeo": {"n_trees": N_TREES_LOEO, "fixed_params": LOEO_PARAMS, "episodes": len(episodes)},
    "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__,
    "warning": "Synthetic-data evidence only; 4 held-out test episodes; overlapping windows; voltage shift not handled; step-vs-ramp average magnitude differs.",
}
with open(OUT / "nb5_v2_config.json", "w", encoding="utf-8") as f:
    json.dump(config_out, f, indent=2, default=str)

print("\nSaved NB5 v2 outputs to:", OUT)
print("Primary interpretation rules:")
print(" - W is the only regime that tests generalisation to unseen cooling/step variants.")
print(" - ALL is the reference for what is learnable when variant types occur in training.")
print(" - Treat episode-level paired LOEO comparisons as the robustness view; four test episodes are too few for strong claims.")
print(" - Do not claim F is superior unless the pattern is consistent across conditions and episode-level uncertainty.")

PASS  refit reproduces W/A/baseline test probabilities exactly

Saved NB5 v2 outputs to: C:\AIoT_Research\outputs\notebook05_v2
Primary interpretation rules:
 - W is the only regime that tests generalisation to unseen cooling/step variants.
 - ALL is the reference for what is learnable when variant types occur in training.
 - Treat episode-level paired LOEO comparisons as the robustness view; four test episodes are too few for strong claims.
 - Do not claim F is superior unless the pattern is consistent across conditions and episode-level uncertainty.
